In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

model = ChatOllama(model="llama3.2", temperature=0.3)

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to gather information about the fictional city of Lunapolis, specifically its capital, weather, population of cheese miners, and potential labor strikes.\n\n## SUMMARY\nThe conversation revolves around the capital of the moon, Lunapolis, and its inhabitants, the cheese miners. The user inquired about the capital, weather, and population of cheese miners, and the AI provided responses. The user also asked if the cheese miners' union would strike, which the AI confirmed due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nDetermine the relevance of the information provided and decide on the next course of action.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='c3a0aca8-2890-44e1-88f3-3edb141bc663'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to gather information about the fictional city of Lunapolis, specifically its capital, weather, population of cheese miners, and potential labor strikes.

## SUMMARY
The conversation revolves around the capital of the moon, Lunapolis, and its inhabitants, the cheese miners. The user inquired about the capital, weather, and population of cheese miners, and the AI provided responses. The user also asked if the cheese miners' union would strike, which the AI confirmed due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
Determine the relevance of the information provided and decide on the next course of action.


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='00cd8947-2558-497c-9690-1d8de4938e95'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='de375d71-abeb-45f0-aa10-42a4cee56aa8', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='8ec5e1f3-983a-423d-b270-5679bff8a5fa'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='e2bd7b62-4c56-4e1f-a8e3-06b4bee3206e', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='5671b297-fcec-46e5-b74c-439bd693dca5'),
              AIMessage(content="That's a good question, but I'm not sure what kind of device y

In [8]:
print(response["messages"][-1].content)

That's a good question, but I'm not sure what kind of device you're referring to. If you're talking about a laptop or a smartphone, it's possible that the device has overheated and shut down. If that's the case, let it cool down for a bit before trying to turn it back on.

If the device is still plugged in and not turning on, here are a few more steps you can try:

1. Press and hold the power button for 10-15 seconds to see if it will turn on.
2. Try charging the device for at least 30 minutes to see if it will turn on.
3. Check for any physical damage, such as water or dust, that may be preventing the device from turning on.
4. If the device is a laptop, try pressing the power button and the volume down button at the same time to see if it will turn on.

If none of these steps work, it's possible that there's a hardware issue with the device and you may need to contact the manufacturer or a repair service for further assistance.

Can you tell me what kind of device you're trying to tu